# Multimodal GIF Generator — Single MP4 Input

Extracts **video**, **audio** and **text (transcript)** signals from a single `.mp4` file and produces three **time-synchronised GIFs** that all play at the original video's speed:

| GIF | FPS | Content |
|-----|-----|---------|
| `gif1_video_slices.gif` | **~30 fps** | Raw video frames (ffmpeg palette-optimised) |
| `gif2_audio_waveform.gif` | **~30 fps** | Scrolling time-vs-amplitude waveform with playhead and word-region overlays |
| `gif3_text_display.gif` | **~30 fps** | Large-text word-by-word highlight with per-word progress bars and timeline ruler |

### Transcript extraction pipeline
1. **ffmpeg** extracts the embedded AAC audio stream to 16 kHz mono WAV.
2. **Short-time energy VAD** (10 ms frames, 5 ms hop) locates voiced regions and merges micro-gaps < 80 ms to segment individual words.
3. The CREMA-D sentence code embedded in the filename (`DFA`, `IEO`, `TAI`, …) is looked up in a built-in dictionary to retrieve the known sentence; VAD regions are then mapped onto those words in order.  
   → If the filename is not a CREMA-D clip, a fallback generic proportional split is used instead.

> All three GIFs share the **same frame count and frame duration** so they stay in perfect sync when played side-by-side.


## Cell 1 — Imports

In [ ]:
import os, io, math, subprocess, json, time, textwrap
import numpy as np
import scipy.io.wavfile as wavfile
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

print("All imports OK")
for lib, obj in [("numpy", np), ("opencv", cv2), ("matplotlib", matplotlib)]:
    print(f"  {lib:<12} {obj.__version__}")


## Cell 2 — Configuration  *(edit here)*

In [ ]:
# ── Input ─────────────────────────────────────────────────────────────────────
MP4_PATH = "/mnt/user-data/uploads/1001_DFA_ANG_XX_01.mp4"   # ← change this path

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR  = "."                          # output directory
OUT_GIF1 = os.path.join(OUT_DIR, "gif1_video_slices.gif")
OUT_GIF2 = os.path.join(OUT_DIR, "gif2_audio_waveform.gif")
OUT_GIF3 = os.path.join(OUT_DIR, "gif3_text_display.gif")

# ── Audio extraction ──────────────────────────────────────────────────────────
WAV_TMP      = "/tmp/_notebook_audio.wav"
AUDIO_SR_OUT = 16000         # resample to 16 kHz for VAD

# ── GIF speed ─────────────────────────────────────────────────────────────────
# GIF_FPS will be set automatically from the video's actual frame rate.
# FRAME_MS is the GIF frame delay in milliseconds (33 ≈ 30 fps).
# Overwrite here to force a different speed, e.g. FRAME_MS = 66 for half speed.
FRAME_MS_OVERRIDE = None     # set to an integer (ms) to override, or None for auto

# ── Resolution ────────────────────────────────────────────────────────────────
OUTPUT_SIZE = None           # None = keep original video resolution; or (w, h) e.g. (320, 240)

# ── Visual palette ────────────────────────────────────────────────────────────
BG_COLOR        = "#0D1B2A"
ACCENT_COLOR    = "#38BDF8"
WAVE_COLOR      = "#34D399"
TEXT_FG_COLOR   = "#E2E8F0"
HIGHLIGHT_COLOR = "#FBBF24"  # amber — currently spoken word

print("Configuration loaded.")
print(f"  Input  : {MP4_PATH}")
print(f"  Output : {OUT_GIF1}, {OUT_GIF2}, {OUT_GIF3}")


## Cell 3 — CREMA-D sentence dictionary

In [ ]:
# Complete CREMA-D 12-sentence lookup (code → text).
# Used to recover the known transcript from the filename without needing a model download.
CREMAD_SENTENCES = {
    "IEO": "It's eleven o'clock",
    "TIE": "That is exactly what happened",
    "IOM": "I'm on my way to the meeting",
    "IWW": "I wonder what this is about",
    "TAI": "The airplane is almost full",
    "MTI": "Maybe tomorrow it will be cold",
    "IWL": "I would like a new alarm clock",
    "ITH": "I think I have a doctor's appointment",
    "DFA": "Don't forget a jacket",
    "ITS": "I'm sorry",
    "TSI": "The surface is slippery",
    "WSI": "We'll stop in a couple of minutes",
}

# CREMA-D emotion codes (for display badge)
CREMAD_EMOTIONS = {
    "ANG": "ANGRY",  "DIS": "DISGUST", "FEA": "FEAR",
    "HAP": "HAPPY",  "NEU": "NEUTRAL", "SAD": "SAD",
}

def parse_cremad_filename(path):
    """
    Parse a CREMA-D filename like  1001_DFA_ANG_XX_01.mp4
    Returns (sentence_code, transcript, emotion_label) or (None, None, None).
    """
    stem = os.path.splitext(os.path.basename(path))[0]
    parts = stem.split("_")
    if len(parts) >= 3:
        s_code  = parts[1].upper()
        e_code  = parts[2].upper()
        sentence = CREMAD_SENTENCES.get(s_code)
        emotion  = CREMAD_EMOTIONS.get(e_code, e_code)
        return s_code, sentence, emotion
    return None, None, None

s_code, sentence, emotion = parse_cremad_filename(MP4_PATH)
if sentence:
    print(f"CREMA-D file detected!")
    print(f"  Sentence code : {s_code}")
    print(f"  Transcript    : \"{sentence}\"")
    print(f"  Emotion       : {emotion}")
else:
    print("Non-CREMA-D file — will use VAD-only word segmentation.")
    sentence = None; emotion = "UNKNOWN"


## Cell 4 — Extract audio & detect word boundaries

In [ ]:
# ── 4a. Extract audio from MP4 ───────────────────────────────────────────────
print("Extracting audio from MP4…")
subprocess.run(
    ["ffmpeg", "-y", "-i", MP4_PATH,
     "-ac", "1", "-ar", str(AUDIO_SR_OUT), "-vn", WAV_TMP],
    capture_output=True, check=True
)
audio_sr, audio_raw = wavfile.read(WAV_TMP)
if audio_raw.ndim == 2:
    audio_raw = audio_raw.mean(axis=1)
audio_data = audio_raw.astype(np.float32)
pk = np.abs(audio_data).max()
if pk > 0:
    audio_data /= pk                 # normalise to [-1, 1]
audio_duration = len(audio_data) / audio_sr
print(f"  Audio : {audio_sr} Hz  |  {audio_duration:.3f}s  |  {len(audio_data)} samples")

# ── 4b. Get video properties ─────────────────────────────────────────────────
cap = cv2.VideoCapture(MP4_PATH)
assert cap.isOpened(), f"Cannot open video: {MP4_PATH}"
video_fps      = cap.get(cv2.CAP_PROP_FPS)
n_vid_frames   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
vid_w          = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
vid_h          = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_duration = n_vid_frames / video_fps
cap.release()
media_duration = min(video_duration, audio_duration)
print(f"  Video : {video_fps:.3f} fps  |  {n_vid_frames} frames  |  {video_duration:.3f}s")
print(f"  Shared duration: {media_duration:.3f}s")

# ── 4c. Output resolution & frame timing ────────────────────────────────────
W, H = OUTPUT_SIZE if OUTPUT_SIZE else (vid_w, vid_h)
FRAME_MS = FRAME_MS_OVERRIDE if FRAME_MS_OVERRIDE else round(1000 / video_fps)
print(f"  Output : {W}×{H}  |  {FRAME_MS} ms/frame (~{1000/FRAME_MS:.1f} fps)")

# ── 4d. Short-time energy VAD ────────────────────────────────────────────────
print("\nRunning short-time energy VAD…")
FL = int(0.010 * audio_sr)     # 10 ms frame length
HL = int(0.005 * audio_sr)     # 5 ms hop
energies, times = [], []
for i in range(0, len(audio_data) - FL, HL):
    energies.append(float(np.sqrt(np.mean(audio_data[i:i+FL]**2))))
    times.append(i / audio_sr)
energies = np.array(energies); times = np.array(times)

# Smooth with 30 ms window then threshold
K = max(1, int(0.030 / (HL / audio_sr)))
smoothed  = np.convolve(energies, np.ones(K)/K, mode="same")
threshold = smoothed.max() * 0.04
voiced    = smoothed > threshold
trans     = np.diff(voiced.astype(int))
onsets    = times[np.where(trans ==  1)[0]]
offsets   = times[np.where(trans == -1)[0]]

# Merge regions with gap < 80 ms
regions = list(zip(onsets[:len(offsets)], offsets))
merged  = [list(regions[0])] if regions else []
for s, e in regions[1:]:
    if s - merged[-1][1] < 0.08:
        merged[-1][1] = e
    else:
        merged.append([s, e])
merged = [(float(s), float(e)) for s, e in merged]
print(f"  Detected {len(merged)} voiced region(s):")
for s, e in merged:
    print(f"    {s:.3f}s – {e:.3f}s  ({e-s:.3f}s)")

# ── 4e. Map words onto voiced regions ───────────────────────────────────────
_, known_transcript, emotion = parse_cremad_filename(MP4_PATH)
if known_transcript:
    known_words = known_transcript.split()
else:
    # For non-CREMA-D files, derive words from voiced region count
    known_words = [f"word{i+1}" for i in range(max(len(merged), 1))]

word_timestamps = []
if len(merged) >= len(known_words):
    # Direct mapping: one region per word
    for i, word in enumerate(known_words):
        word_timestamps.append({"word": word,
                                 "start": merged[i][0],
                                 "end":   merged[i][1]})
elif len(merged) > 0:
    # Fewer regions than words: distribute proportionally within total speech span
    speech_start = merged[0][0]
    speech_end   = merged[-1][1]
    span         = speech_end - speech_start
    n            = len(known_words)
    for i, word in enumerate(known_words):
        ws = speech_start + i/n * span
        we = speech_start + (i+1)/n * span
        word_timestamps.append({"word": word, "start": float(ws), "end": float(we)})
else:
    # No voiced regions found: spread evenly
    n = len(known_words)
    for i, word in enumerate(known_words):
        word_timestamps.append({"word": word,
                                 "start": i/n * media_duration,
                                 "end":   (i+1)/n * media_duration})

transcript = " ".join(w["word"] for w in word_timestamps)
print(f"\nTranscript : \"{transcript}\"")
print(f"Emotion    : {emotion}")
print("\nWord timestamps:")
for w in word_timestamps:
    print(f"  {w['word']:12s}  {w['start']:.3f}s – {w['end']:.3f}s")


## Cell 5 — Helper functions

In [ ]:
def hex_to_rgb(h):
    """'#RRGGBB' → (R, G, B) ints."""
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def hex_to_float(h):
    """'#RRGGBB' → (r, g, b) floats for matplotlib."""
    return tuple(c/255 for c in hex_to_rgb(h))

def best_font(paths, size):
    """Load the first available TrueType font from *paths*, fallback to default."""
    for p in paths:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

def emotion_colour(emotion_label):
    """Return a hex colour matching the CREMA-D emotion."""
    return {
        "ANGRY":   "#F87171", "DISGUST": "#C084FC",
        "FEAR":    "#FB923C", "HAPPY":   "#4ADE80",
        "NEUTRAL": "#94A3B8", "SAD":     "#60A5FA",
    }.get(emotion_label.upper(), "#E2E8F0")

print("Helper functions defined.")


## Cell 6 — GIF 1: Video frames  *(ffmpeg)*

In [ ]:
print(f"[GIF 1] Encoding {n_vid_frames} video frames @ ~{1000/FRAME_MS:.0f} fps via ffmpeg …")
t0 = time.time()

palette_path = "/tmp/_gif1_palette.png"

# Pass 1 — generate perceptual colour palette from the full clip
subprocess.run(
    ["ffmpeg", "-y", "-i", MP4_PATH,
     "-vf", f"fps={video_fps:.4f},scale={W}:{H}:flags=lanczos,"
            "palettegen=max_colors=256:stats_mode=full",
     palette_path],
    capture_output=True, check=True
)

# Pass 2 — encode GIF using that palette with Bayer dithering
r = subprocess.run(
    ["ffmpeg", "-y", "-i", MP4_PATH, "-i", palette_path,
     "-lavfi",
     f"fps={video_fps:.4f},scale={W}:{H}:flags=lanczos[x];"
     "[x][1:v]paletteuse=dither=bayer:bayer_scale=5:diff_mode=rectangle",
     OUT_GIF1],
    capture_output=True, text=True
)
assert os.path.exists(OUT_GIF1) and os.path.getsize(OUT_GIF1) > 0, \
    f"ffmpeg GIF1 failed:\n{r.stderr[-500:]}"

print(f"  ✓  {OUT_GIF1}")
print(f"     {os.path.getsize(OUT_GIF1)//1024} KB  |  {n_vid_frames} frames  |  {time.time()-t0:.1f}s")


## Cell 7 — GIF 2: Audio waveform

In [ ]:
print(f"[GIF 2] Rendering {n_vid_frames} waveform frames …")
t0 = time.time()

dpi   = 100
bg_f  = hex_to_float(BG_COLOR)
wv_f  = hex_to_float(WAVE_COLOR)
acc_f = hex_to_float(ACCENT_COLOR)
txt_f = hex_to_float(TEXT_FG_COLOR)
hi_f  = hex_to_float(HIGHLIGHT_COLOR)

# Pre-compute full waveform downsampled for background
full_t   = np.linspace(0, audio_duration, len(audio_data))
step_bg  = max(1, len(audio_data) // 2000)

gif2_frames = []
for fi in range(n_vid_frames):
    t_now = fi / video_fps

    # 500 ms rolling window around the playhead
    win_s = max(0, t_now - 0.45)
    win_e = min(t_now + 0.05, audio_duration)
    smp_s = int(win_s * audio_sr)
    smp_e = int(win_e * audio_sr)
    seg   = audio_data[smp_s:smp_e]
    t_seg = np.linspace(win_s, win_e, max(len(seg), 1))

    fig, ax = plt.subplots(figsize=(W/dpi, H/dpi), dpi=dpi)
    fig.patch.set_facecolor(bg_f)
    ax.set_facecolor(bg_f)

    # Dimmed full waveform (context)
    ax.plot(full_t[::step_bg], audio_data[::step_bg],
            color=wv_f, linewidth=0.4, alpha=0.18)

    # Bright current window
    step_w = max(1, len(seg) // 600)
    if len(seg) > 0:
        ax.plot(t_seg[::step_w], seg[::step_w],
                color=wv_f, linewidth=1.3, alpha=0.95)
        ax.fill_between(t_seg[::step_w], seg[::step_w], 0,
                        color=wv_f, alpha=0.22)

    # Playhead vertical line
    ax.axvline(x=t_now, color=acc_f, linewidth=1.6, alpha=0.9, zorder=5)

    # Word-region shading + labels
    for w in word_timestamps:
        ws, we = w["start"], w["end"]
        active = ws <= t_now <= we
        ax.axvspan(ws, we,
                   color=hi_f, alpha=0.15 if active else 0.06, zorder=1)
        ax.text((ws+we)/2, -0.88, w["word"],
                ha="center", va="bottom", fontsize=7,
                color=hi_f, alpha=0.9 if active else 0.45)

    ax.set_xlim(0, audio_duration)
    ax.set_ylim(-1.15, 1.15)
    for sp in ax.spines.values():
        sp.set_edgecolor(acc_f); sp.set_linewidth(0.6)
    ax.tick_params(colors=txt_f, labelsize=7)
    ax.set_xlabel("Time (s)", fontsize=8, color=txt_f)
    ax.set_ylabel("Amplitude", fontsize=8, color=txt_f)
    ax.set_title(f"Audio Waveform   t = {t_now:.3f}s   |   {transcript}",
                 color=txt_f, fontsize=8.5, pad=4)

    # Progress bar (thin strip at top)
    prog = min(t_now / media_duration, 1.0)
    ax.axhline(y=1.08, xmin=0, xmax=1,
               color=hex_to_float("#1E293B"), linewidth=4, solid_capstyle="butt")
    ax.axhline(y=1.08, xmin=0, xmax=prog,
               color=acc_f, linewidth=4, solid_capstyle="butt")

    fig.tight_layout(pad=0.35)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi)
    buf.seek(0)
    gif2_frames.append(
        Image.open(buf).convert("RGB").resize((W, H), Image.LANCZOS).copy()
    )
    plt.close(fig)

gif2_frames[0].save(
    OUT_GIF2, save_all=True, append_images=gif2_frames[1:],
    loop=0, duration=FRAME_MS, optimize=False
)
print(f"  ✓  {OUT_GIF2}")
print(f"     {os.path.getsize(OUT_GIF2)//1024} KB  |  {len(gif2_frames)} frames  |  {time.time()-t0:.1f}s")


## Cell 8 — GIF 3: Text / transcript display

In [ ]:
print(f"[GIF 3] Rendering {n_vid_frames} text frames …")
t0 = time.time()

font_word  = best_font([
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"], 28)
font_small = best_font([
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf"], 11)

bg_rgb  = hex_to_rgb(BG_COLOR)
acc_rgb = hex_to_rgb(ACCENT_COLOR)
hi_rgb  = hex_to_rgb(HIGHLIGHT_COLOR)
dim_rgb = (55, 75, 100)      # not-yet-spoken
spo_rgb = (110, 150, 185)    # already-spoken
emo_rgb = hex_to_rgb(emotion_colour(emotion))

TITLE_H = 38
MARGIN  = 24
GAP     = 20   # pixels between words

# Pre-compute word pixel widths (shared dummy draw)
_dummy_img  = Image.new("RGB", (W, H))
_dummy_draw = ImageDraw.Draw(_dummy_img)
word_widths = {}
for w in word_timestamps:
    bb = _dummy_draw.textbbox((0, 0), w["word"], font=font_word)
    word_widths[w["word"]] = bb[2] - bb[0]

total_row_w = sum(word_widths.values()) + GAP * (len(word_timestamps) - 1)
x_start     = max(MARGIN, (W - total_row_w) // 2)
y_word      = H // 2 - 35
y_bars      = y_word + 68

gif3_frames = []
for fi in range(n_vid_frames):
    t_now = fi / video_fps

    img  = Image.new("RGB", (W, H), bg_rgb)
    draw = ImageDraw.Draw(img)

    # ── Title bar ────────────────────────────────────────────────────────────
    draw.rectangle([(0, 0), (W, TITLE_H)], fill=(15, 32, 54))
    draw.text((MARGIN, 10),
              f"Transcript   t = {t_now:.3f}s  /  {media_duration:.3f}s",
              fill=acc_rgb, font=font_small)
    # Emotion badge (right side)
    draw.text((W - 90, 10), emotion, fill=emo_rgb, font=font_small)

    # ── Words ─────────────────────────────────────────────────────────────────
    x = x_start
    for w in word_timestamps:
        ws, we   = w["start"], w["end"]
        ww       = word_widths[w["word"]]
        active   = ws <= t_now <= we
        past     = t_now > we

        colour = hi_rgb if active else (spo_rgb if past else dim_rgb)

        # Soft glow behind active word
        if active:
            for dx, dy in [(-1,-1),(1,-1),(-1,1),(1,1)]:
                draw.text((x+dx, y_word+dy), w["word"],
                          fill=(100, 75, 15), font=font_word)

        draw.text((x, y_word), w["word"], fill=colour, font=font_word)

        # Underline for active word
        if active:
            bb = draw.textbbox((x, y_word), w["word"], font=font_word)
            draw.rectangle([(bb[0], bb[3]+3), (bb[2], bb[3]+6)], fill=hi_rgb)

        # ── Per-word progress bar (below word) ───────────────────────────────
        bar_h_max = 28
        draw.rectangle([(x, y_bars), (x+ww, y_bars+bar_h_max)],
                       fill=(22, 42, 68))
        if t_now >= we:
            fill_frac = 1.0
        elif t_now >= ws:
            fill_frac = (t_now - ws) / max(we - ws, 1e-6)
        else:
            fill_frac = 0.0
        if fill_frac > 0:
            bar_col = hi_rgb if fill_frac < 1.0 else spo_rgb
            draw.rectangle([(x, y_bars), (x + int(ww * fill_frac), y_bars+bar_h_max)],
                           fill=bar_col)
        # Duration label inside bar
        dur_txt = f"{we-ws:.2f}s"
        draw.text((x+3, y_bars + bar_h_max//2 - 5),
                  dur_txt, fill=(160, 190, 210), font=font_small)

        x += ww + GAP

    # ── Timeline ruler ────────────────────────────────────────────────────────
    ruler_y = H - 38
    ruler_w = W - 2*MARGIN
    draw.rectangle([(MARGIN, ruler_y), (MARGIN+ruler_w, ruler_y+4)],
                   fill=(28, 48, 72))

    # Word spans on ruler (coloured blocks)
    for w in word_timestamps:
        rx_s = MARGIN + int(w["start"] / media_duration * ruler_w)
        rx_e = MARGIN + int(w["end"]   / media_duration * ruler_w)
        draw.rectangle([(rx_s, ruler_y-2), (rx_e, ruler_y+6)],
                       fill=acc_rgb)

    # Playhead on ruler
    rx_now = MARGIN + int(min(t_now / media_duration, 1.0) * ruler_w)
    draw.rectangle([(rx_now-2, ruler_y-7), (rx_now+2, ruler_y+11)],
                   fill=hi_rgb)

    # Start / end time labels
    draw.text((MARGIN, ruler_y+9), "0.0s", fill=dim_rgb, font=font_small)
    draw.text((W-MARGIN-28, ruler_y+9), f"{media_duration:.2f}s",
              fill=dim_rgb, font=font_small)

    gif3_frames.append(img)

gif3_frames[0].save(
    OUT_GIF3, save_all=True, append_images=gif3_frames[1:],
    loop=0, duration=FRAME_MS, optimize=False
)
print(f"  ✓  {OUT_GIF3}")
print(f"     {os.path.getsize(OUT_GIF3)//1024} KB  |  {len(gif3_frames)} frames  |  {time.time()-t0:.1f}s")


## Cell 9 — Summary & first-frame preview

In [ ]:
print("─" * 60)
print("OUTPUT SUMMARY")
print("─" * 60)
total_kb = 0
for path, label in [
    (OUT_GIF1, "GIF1 – Video"),
    (OUT_GIF2, "GIF2 – Waveform"),
    (OUT_GIF3, "GIF3 – Text"),
]:
    kb = os.path.getsize(path) // 1024
    total_kb += kb
    g  = Image.open(path)
    nf = getattr(g, "n_frames", 1)
    print(f"  {label:<20}  {kb:>5} KB   {nf} frames   {FRAME_MS} ms/frame")
print(f"  {'TOTAL':<20}  {total_kb:>5} KB")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, path, title in zip(
    axes,
    [OUT_GIF1, OUT_GIF2, OUT_GIF3],
    ["GIF 1\nVideo", "GIF 2\nAudio Waveform", "GIF 3\nTranscript"],
):
    ax.imshow(np.array(Image.open(path)))
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.suptitle(
    f'{os.path.basename(MP4_PATH)}   |   "{transcript}"   |   {emotion}',
    fontsize=9, y=1.01
)
plt.tight_layout()
plt.savefig("preview_first_frames.png", dpi=90, bbox_inches="tight")
plt.show()
print("\nFirst-frame preview saved → preview_first_frames.png")


## Cell 10 — Play all three GIFs inline *(Jupyter only)*

In [ ]:
from IPython.display import HTML
import base64

def gif_inline(path, label, width=None):
    w = width or W
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    return (
        f'<div style="display:inline-block;margin:6px;text-align:center;">'
        f'<p style="font-family:monospace;font-size:12px;color:#38BDF8;margin:0 0 4px">'
        f'{label}</p>'
        f'<img src="data:image/gif;base64,{data}" width="{w}" '
        f'style="border:1px solid #38BDF8;border-radius:4px"/>'
        f'</div>'
    )

HTML(
    '<div style="background:#0D1B2A;padding:14px;border-radius:6px">'
    + gif_inline(OUT_GIF1, f"GIF 1 — Video ({n_vid_frames} frames, ~{1000/FRAME_MS:.0f} fps)")
    + gif_inline(OUT_GIF2, f"GIF 2 — Audio Waveform ({n_vid_frames} frames)")
    + gif_inline(OUT_GIF3, f"GIF 3 — Transcript: \"{transcript}\" [{emotion}]")
    + "</div>"
)
